# Phase 2 — Train the Stage-1 Baseline Estimator on Colab (T4 GPU)

**Run this notebook with the Colab runtime kernel** (Google Colab VS Code extension → select a GPU runtime), so all cells execute on Google's T4, not your Mac.

**Prerequisites (one-time):**
1. Colab extension installed; runtime set to **GPU (T4)**.
2. The image set zipped and uploaded to your Google Drive (see `EDIT ME` cell).
3. The repo pushed to GitHub (public, or private with a token).

The flow: mount Drive → clone repo (code + split CSVs) → unzip images from Drive → train on GPU → copy checkpoint back to Drive.


## 1. Confirm we're on a GPU runtime

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch runtime to GPU!")

## 2. Mount Google Drive (for the dataset zip and to save checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. EDIT ME — paths and repo
Set these three values, then run every cell below top-to-bottom.

In [ ]:
REPO_URL    = "https://github.com/amoiz69/pv-fault-classification.git"
DRIVE_IMAGES_ZIP = "/content/drive/MyDrive/pv-fault/images.zip"                  # <-- where you uploaded the zip
DRIVE_CKPT_DIR   = "/content/drive/MyDrive/pv-fault/checkpoints"                 # <-- where checkpoints persist

REPO_DIR = "/content/pv-fault-classification"
# For a PRIVATE repo, instead use a token URL:
# REPO_URL = "https://<TOKEN>@github.com/<YOUR_USERNAME>/pv-fault-classification.git" 

## 4. Clone the repo (brings src/ + committed split CSVs + module_metadata.json)

In [ ]:
import os, shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)              # always start from the latest pushed code
!git clone "$REPO_URL" "$REPO_DIR"
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls data && echo "--- splits ---" && ls data/splits

## 5. Unzip the images from Drive into `data/`
The 20k JPEGs are gitignored (not in the repo), so they come from your Drive zip.
The zip should contain an `images/` folder; it unzips to `data/images/`.

In [ ]:
!unzip -q -o "$DRIVE_IMAGES_ZIP" -d data/
import glob
n = len(glob.glob("data/images/*.jpg"))
print(f"{n} images present")
assert n == 20000, "expected 20000 images — check the zip contents/structure"


## 6. (Optional) dependencies
Colab already ships torch, numpy, pandas, scikit-learn, Pillow, pyyaml — enough
for the estimator. Only run this if an import fails; avoid reinstalling torch.

In [ ]:
# !pip install -q pyyaml

## 7. Train on the T4
Reads `configs/estimator.yaml`; trains on No-Anomaly only; saves `checkpoints/estimator.pt`.

In [ ]:
!python scripts/train_estimator.py --config configs/estimator.yaml

## 8. Persist the checkpoint to Drive
Colab runtimes are ephemeral — copy the result out before disconnecting.

In [ ]:
import os, shutil
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
src = os.path.join(REPO_DIR, "checkpoints", "estimator.pt")
dst = os.path.join(DRIVE_CKPT_DIR, "estimator.pt")
shutil.copy(src, dst)
print("saved to Drive:", dst)
ckpt = torch.load(src, map_location="cpu")
print("best val MAE:", round(ckpt["val_mae"], 4))

## 9. Quick sanity: residual on healthy vs fault (runs on GPU here)
Healthy residuals should sit near 0; fault residuals should show a non-zero spike.
This is the proof the estimator removes baseline without erasing faults.

In [ ]:
import sys; sys.path.insert(0, REPO_DIR)
from src.dataset import InfraredSolarDataset
from src.estimator import BaselineEstimator
from src.residual import generate_residual, residual_stats
from torch.utils.data import DataLoader

m = BaselineEstimator(width=ckpt["width"]); m.load_state_dict(ckpt["model_state"]); m.eval()

healthy = InfraredSolarDataset("data", "data/splits/noanomaly_val.csv")
faults  = InfraredSolarDataset("data", "data/splits/fault_val.csv")
hb, _ = next(iter(DataLoader(healthy, batch_size=256)))
fb, _ = next(iter(DataLoader(faults,  batch_size=256)))
print("healthy residual:", {k: round(v,4) for k,v in residual_stats(generate_residual(hb, m)).items()})
print("fault   residual:", {k: round(v,4) for k,v in residual_stats(generate_residual(fb, m)).items()})

## Next steps
- Download `estimator.pt` from Drive into your local `checkpoints/` (or sync Drive).
- Run `notebooks/02_baseline_estimator.ipynb` locally (inference only, no training) for the full residual visualization and the predicted-vs-actual plots that go in the paper.
